In [ ]:
@file:DependsOn("org.postgresql:postgresql:42.7.4")

In [ ]:
%use dataframe
%use kandy

In [ ]:
import bps.jdbc.getConfigFromResource

//bps.budget.model.DraftStatus.clearing
// FIXME project isn't building so I can't use classes from budgetDao
val URL = "jdbc:postgresql://localhost:5432/budget?currentSchema=budget"
val USER_NAME = "budget"
val PASSWORD = "budget"

getConfigFromResource("")

val dbConfig = DbConnectionConfig(URL, USER_NAME, PASSWORD)

In [ ]:
val budgetAccessId = "7e8366d3-6710-4342-a3ea-26282ddb65f6"
val budgetName = "Benji's Budget"

In [ ]:
import java.sql.PreparedStatement

dbConfig.connection.apply {

prepareStatement(
    """
                            select b.general_account_name, b.id as budget_id, u.id as user_id, ba.time_zone, ba.budget_name, ba.analytics_start, a.id as general_account_id
                            from budgets b
                                join budget_access ba on b.id = ba.budget_id
                                join users u on u.id = ba.user_id
                                join accounts a
                                    on a.name = b.general_account_name
                                    and a.type = 'category'
                                    and a.budget_id = b.id
                            where ba.budget_name = ?
                                and u.login = ?
                        """.trimIndent(),
)
    .use { getBudget: PreparedStatement ->
        getBudget.setString(1, budgetName)
        getBudget.setString(2, userName)
        getBudget.executeQuery()
            .use { result: ResultSet ->
                if (result.next()) {
                    BudgetDataInfo(
                        generalAccountId = result.getUuid("general_account_id")!!,
                        timeZone = result.getString("time_zone")
                            ?.let { timeZone -> TimeZone.of(timeZone) }
                            ?: TimeZone.currentSystemDefault(),
                        analyticsStart = result.getInstant("analytics_start"),
                        budgetName = budgetName,
                        budgetId = result.getUuid("budget_id")!!,
                        userName = userName,
                        userId = result.getUuid("user_id")!!,
                    )
                } else
                    throw DataConfigurationException("Budget data not found for name: $budgetName")
            }
    }

In [ ]:
val sqlQuery = """
                |select t.timestamp_utc, i.amount
                |from transaction_items i
                |join transactions t
                |    on i.transaction_id = t.id
                |    and i.budget_id = t.budget_id
                |join accounts a
                |    on i.account_id = a.id
                |    and i.budget_id = a.budget_id
                |where t.type = 'expense'
                |  and i.amount < 0
                |  and a.type = 'category'
                |  and t.timestamp_utc >= '2024-11-01T00:00:00.000Z'
                |  and t.timestamp_utc < ?
                |  and i.budget_id = ?
                |order by t.timestamp_utc asc
            """.trimMargin()
val spending = DataFrame.readSqlQuery(dbConfig, sqlQuery)
spending.schema()

In [ ]:
//                |  ${if (options.endDateLimited) "and t.timestamp_utc < ?" else ""}